In [0]:
import os
import sys

sys.path.append(os.path.abspath("../../src"))

from pyspark.sql import functions as F
from pipeline.silver.telemetry import (
    clean_telemetry,
    assert_positions_valid,
    add_sections,
    assert_sections_matched,
    add_laps,
)


In [0]:
SOURCE = "motorsport.bronze.telemetry"
SESSIONS = "motorsport.silver.sessions"
SECTIONS = "motorsport.silver.track_sections"
TARGET = "motorsport.silver.telemetry"

NON_SENSOR = {
    "session_id", "timestamp", "_source_file", "_ingested_at",
    "track_id", "section_number", "lap",
}

In [0]:
bronze = spark.table(SOURCE)

In [0]:
if spark.catalog.tableExists(TARGET):
    done = spark.table(TARGET).select("session_id").distinct()
    new = bronze.join(done, "session_id", "left_anti")
else:
    new = bronze

if new.limit(1).count() == 0:
    dbutils.notebook.exit("no new sessions")

In [0]:
sensor_cols = [c for c in new.columns if c not in NON_SENSOR]

cleaned = clean_telemetry(new, sensor_cols)

# track_id lives on the session, not in the Parquet
sessions = spark.table(SESSIONS).select("session_id", "track_id")
with_track = cleaned.join(sessions, "session_id", "left")

missing = with_track.filter(F.col("track_id").isNull()).select("session_id").distinct()
if missing.limit(1).count() > 0:
    raise ValueError(
        f"sessions missing from silver.sessions: {[r[0] for r in missing.collect()]}"
    )

assert_positions_valid(with_track)


In [0]:
sections = spark.table(SECTIONS).select(
    "track_id",
    "section_number",
    F.col("start_coordinate.x").alias("start_x"),
    F.col("start_coordinate.y").alias("start_y"),
)

sectioned = add_sections(with_track, sections)
assert_sections_matched(sectioned)
result = add_laps(sectioned)

In [0]:
result.write.mode("append").saveAsTable(TARGET)